# Notebook 01 — Eksplorasi Awal Dataset Diare Balita

Notebook ini disusun sebagai tahap awal **Exploratory Data Analysis (EDA)** untuk proyek Data Analytics dan Data Science.

## Tujuan
- Membaca dataset secara terstruktur dan dapat dikonfigurasi.
- Memahami ukuran, struktur, tipe data, dan karakteristik variabel.
- Mengidentifikasi missing values, duplikasi, nilai konstan, dan potensi anomali format.
- Menghasilkan ringkasan statistik dan visualisasi awal secara otomatis.
- Menyimpan ringkasan hasil eksplorasi agar dapat digunakan pada tahap berikutnya.

## Cara menggunakan
1. Letakkan dataset di `data/raw/` pada root repository.
2. Jalankan notebook dari atas ke bawah.
3. Jika folder berisi lebih dari satu dataset, isi `DATASET_FILE` pada bagian konfigurasi dengan nama file yang ingin dianalisis.
4. Jangan menggabungkan beberapa file secara otomatis sebelum memastikan bahwa skema dan unit datanya memang sebanding.

Notebook mendukung CSV, Excel (`.xlsx`, `.xls`), dan Parquet. Analisis bersifat eksploratif; hasil asosiasi atau distribusi **bukan bukti sebab-akibat** dan bukan diagnosis klinis.

## 1. Import library

Library utama digunakan untuk pengolahan data, visualisasi, dan pemeriksaan kualitas data. Jika ada library yang belum tersedia, pasang dependensi dari `requirements.txt` proyek atau jalankan instalasi pada cell berikut.

In [ ]:
# Jika diperlukan, aktifkan baris berikut untuk memasang dependensi:
# %pip install pandas numpy matplotlib seaborn openpyxl pyarrow

from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

sns.set_theme(style="whitegrid", palette="Set2")
print("Library berhasil diimpor.")

## 2. Konfigurasi proyek dan pencarian dataset

Pencarian direktori dibuat relatif terhadap lokasi notebook sehingga tidak bergantung pada satu komputer. Notebook mencoba menemukan root proyek dari folder `notebooks/eda/`, lalu mencari file pada `data/raw/`.

**Penting:** jika terdapat beberapa file, tentukan file yang tepat melalui `DATASET_FILE`. Notebook tidak menggabungkan file berbeda secara otomatis karena kolom, periode, atau unit pengukuran bisa berbeda.

In [ ]:
# Ubah hanya jika struktur repository berbeda.
DATASET_FILE = None  # Contoh: "dataset_diare_balita.csv"
SUPPORTED_EXTENSIONS = {".csv", ".xlsx", ".xls", ".parquet"}

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "data" / "raw").exists() or (candidate / "notebooks").exists():
            return candidate
    # Jika notebook dijalankan dari notebooks/eda/, root berada dua tingkat di atas.
    if start.name.lower() == "eda" and start.parent.name.lower() == "notebooks":
        return start.parent.parent
    # Jika dijalankan dari folder kerja biasa, gunakan folder tersebut.
    return start

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "reports" / "eda"

# Kandidat lokasi tambahan membantu saat notebook dijalankan dari lingkungan berbeda.
search_dirs = [RAW_DATA_DIR, NOTEBOOK_DIR, PROJECT_ROOT]
if Path("/mnt/data").exists():
    search_dirs.append(Path("/mnt/data"))

dataset_files = []
seen = set()
for folder in search_dirs:
    if folder.exists():
        for file in folder.iterdir():
            if file.is_file() and file.suffix.lower() in SUPPORTED_EXTENSIONS:
                resolved = file.resolve()
                if resolved not in seen:
                    dataset_files.append(resolved)
                    seen.add(resolved)

dataset_files = sorted(dataset_files, key=lambda x: (x.parent != RAW_DATA_DIR.resolve(), x.name.lower()))

print(f"Root proyek : {PROJECT_ROOT}")
print(f"Folder raw  : {RAW_DATA_DIR}")
print(f"Folder output: {OUTPUT_DIR}")
print("\nDataset yang ditemukan:")
for i, file in enumerate(dataset_files, start=1):
    print(f"{i}. {file}")

if DATASET_FILE:
    matches = [f for f in dataset_files if f.name == DATASET_FILE]
    if not matches:
        raise FileNotFoundError(
            f"File '{DATASET_FILE}' tidak ditemukan. Letakkan file di data/raw/ "
            "atau periksa kembali nama file pada DATASET_FILE."
        )
    DATASET_PATH = matches[0]
else:
    preferred = [f for f in dataset_files if f.parent == RAW_DATA_DIR.resolve()]
    candidates = preferred or dataset_files
    if len(candidates) == 0:
        raise FileNotFoundError(
            "Tidak ada dataset yang didukung. Letakkan CSV/XLSX/XLS/Parquet di data/raw/."
        )
    if len(candidates) > 1:
        raise RuntimeError(
            "Ditemukan lebih dari satu dataset pada lokasi prioritas. "
            "Isi DATASET_FILE dengan nama file yang ingin dianalisis."
        )
    DATASET_PATH = candidates[0]

print(f"\nDataset terpilih: {DATASET_PATH.name}")

## 3. Membaca dataset

Pembacaan data disesuaikan dengan ekstensi file. Untuk CSV, pemisah kolom dideteksi otomatis. Untuk Excel, sheet pertama digunakan secara default; ubah `EXCEL_SHEET` jika diperlukan.

In [ ]:
EXCEL_SHEET = 0  # Bisa diganti dengan nama sheet, misalnya "Data"

def load_dataset(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        # sep=None membantu mendeteksi koma atau titik koma.
        return pd.read_csv(path, sep=None, engine="python")
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path, sheet_name=EXCEL_SHEET)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Format file belum didukung: {suffix}")

df = load_dataset(DATASET_PATH)
if df.empty:
    raise ValueError("Dataset berhasil dibaca, tetapi tidak memiliki baris data.")

# Rapikan spasi di nama kolom tanpa mengubah nama aslinya secara agresif.
df.columns = [str(col).strip() for col in df.columns]

print("Dataset berhasil dibaca.")
print(f"Ukuran dataset: {df.shape[0]:,} baris × {df.shape[1]:,} kolom")
display(df.head())

## 4. Pemeriksaan struktur dataset

Pemeriksaan ini menampilkan ukuran data, nama kolom, tipe data, jumlah nilai unik, serta contoh nilai. Informasi ini membantu menentukan perlakuan yang tepat pada tahap data cleaning dan feature engineering.

In [ ]:
print("Lima baris pertama:")
display(df.head())

print("\nDimensi dataset:")
print(f"Baris: {df.shape[0]:,}")
print(f"Kolom: {df.shape[1]:,}")

structure = pd.DataFrame({
    "kolom": df.columns,
    "tipe_data": [str(df[c].dtype) for c in df.columns],
    "jumlah_non_null": [df[c].notna().sum() for c in df.columns],
    "jumlah_null": [df[c].isna().sum() for c in df.columns],
    "jumlah_unik": [df[c].nunique(dropna=True) for c in df.columns],
    "contoh_nilai": [
        ", ".join(map(str, df[c].dropna().astype(str).unique()[:3]))
        for c in df.columns
    ],
})
display(structure)

## 5. Pemeriksaan kualitas data awal

Pemeriksaan mencakup missing values, baris duplikat, kolom duplikat, kolom konstan, dan nilai kosong berbentuk teks. Pemeriksaan ini hanya menandai potensi masalah; notebook tidak menghapus atau mengubah data sumber.

In [ ]:
# Missing values per kolom
missing = pd.DataFrame({
    "jumlah_missing": df.isna().sum(),
    "persentase_missing": (df.isna().mean() * 100).round(2)
}).sort_values("persentase_missing", ascending=False)
display(missing)

# Duplikasi baris penuh
duplicate_rows = int(df.duplicated().sum())
print(f"Jumlah baris duplikat penuh: {duplicate_rows:,}")

# Kolom dengan satu nilai unik atau kurang
constant_columns = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
print("Kolom konstan:", constant_columns if constant_columns else "Tidak ada")

# String kosong atau hanya spasi (tidak termasuk NaN)
blank_text = {}
for col in df.select_dtypes(include=["object", "string"]).columns:
    blank_text[col] = int(df[col].astype("string").str.strip().eq("").sum())
blank_text = {k: v for k, v in blank_text.items() if v > 0}
print("Sel kosong berbentuk teks:", blank_text if blank_text else "Tidak ditemukan")

# Kolom yang berpotensi menjadi identifier berdasarkan nama dan keunikan.
id_like_columns = [
    c for c in df.columns
    if any(token in c.lower() for token in ["id", "kode", "nomor", "no."])
]
print("Kolom yang berpotensi menjadi identifier:", id_like_columns or "Tidak terdeteksi")

## 6. Statistik deskriptif

Variabel numerik diringkas menggunakan statistik deskriptif. Variabel kategorikal diringkas berdasarkan frekuensi nilai. Statistik ini belum menyatakan apakah nilai tertentu sehat/tidak sehat atau normal/tidak normal; interpretasi klinis memerlukan standar dan konteks pengukuran yang tervalidasi.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category", "bool", "string"]).columns.tolist()

print(f"Variabel numerik ({len(numeric_cols)}): {numeric_cols}")
if numeric_cols:
    display(df[numeric_cols].describe().T)

print(f"\nVariabel kategorikal ({len(categorical_cols)}): {categorical_cols}")
for col in categorical_cols:
    print(f"\nDistribusi: {col}")
    display(
        df[col].value_counts(dropna=False)
          .rename_axis("kategori")
          .reset_index(name="jumlah")
          .head(20)
    )

## 7. Deteksi kandidat kolom target

Notebook mencoba mengenali kolom target berdasarkan nama yang umum digunakan. Deteksi ini hanya saran awal dan tidak otomatis menetapkan target untuk pemodelan.

In [ ]:
TARGET_CANDIDATES = [
    "status diare", "diare", "target", "label", "kelas", "status"
]

def normalize_name(value: str) -> str:
    return " ".join(str(value).lower().replace("_", " ").replace("-", " ").split())

normalized_columns = {normalize_name(c): c for c in df.columns}
target_matches = [
    normalized_columns[name] for name in TARGET_CANDIDATES
    if name in normalized_columns
]
TARGET_COLUMN = target_matches[0] if target_matches else None

if TARGET_COLUMN:
    print(f"Kandidat kolom target: {TARGET_COLUMN}")
    print("Distribusi kandidat target:")
    display(df[TARGET_COLUMN].value_counts(dropna=False).rename("jumlah").to_frame())
else:
    print("Kolom target tidak terdeteksi otomatis. Tentukan secara manual jika diperlukan.")

## 8. Visualisasi eksplorasi awal

Grafik dibuat secara dinamis berdasarkan tipe data:
- Histogram untuk beberapa variabel numerik.
- Boxplot untuk melihat sebaran dan nilai ekstrem secara visual.
- Bar chart untuk beberapa variabel kategorikal.
- Grafik distribusi target jika kandidat target ditemukan.

Jumlah grafik dibatasi agar notebook tetap terbaca pada dataset dengan banyak kolom.

In [ ]:
# 8.1 Histogram variabel numerik
plot_numeric = [c for c in numeric_cols if df[c].nunique(dropna=True) > 1][:8]

if plot_numeric:
    ncols = 2
    nrows = int(np.ceil(len(plot_numeric) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, plot_numeric):
        values = pd.to_numeric(df[col], errors="coerce").dropna()
        ax.hist(values, bins="auto", edgecolor="black", alpha=0.75)
        ax.set_title(f"Distribusi {col}")
        ax.set_xlabel(col)
        ax.set_ylabel("Frekuensi")
    for ax in axes[len(plot_numeric):]:
        ax.remove()
    plt.tight_layout()
    plt.show()
else:
    print("Tidak ada variabel numerik yang bervariasi untuk dibuat histogram.")

# 8.2 Boxplot variabel numerik
if plot_numeric:
    ncols = 2
    nrows = int(np.ceil(len(plot_numeric) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, plot_numeric):
        values = pd.to_numeric(df[col], errors="coerce").dropna()
        ax.boxplot(values, vert=False)
        ax.set_title(f"Boxplot {col}")
        ax.set_xlabel(col)
    for ax in axes[len(plot_numeric):]:
        ax.remove()
    plt.tight_layout()
    plt.show()

# 8.3 Bar chart kategori dengan jumlah kategori terbatas
plot_categorical = [
    c for c in categorical_cols
    if 2 <= df[c].nunique(dropna=True) <= 15
][:8]

if plot_categorical:
    ncols = 2
    nrows = int(np.ceil(len(plot_categorical) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4.5 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, plot_categorical):
        counts = df[col].astype("string").fillna("(Missing)").value_counts().head(15)
        ax.barh(counts.index.astype(str)[::-1], counts.values[::-1], edgecolor="black", alpha=0.75)
        ax.set_title(f"Frekuensi {col}")
        ax.set_xlabel("Jumlah")
        ax.set_ylabel(col)
    for ax in axes[len(plot_categorical):]:
        ax.remove()
    plt.tight_layout()
    plt.show()
else:
    print("Tidak ada variabel kategorikal dengan 2–15 kategori untuk divisualisasikan.")

# 8.4 Distribusi kandidat target
if TARGET_COLUMN:
    counts = df[TARGET_COLUMN].astype("string").fillna("(Missing)").value_counts()
    plt.figure(figsize=(8, 4))
    plt.bar(counts.index.astype(str), counts.values, edgecolor="black", alpha=0.75)
    plt.title(f"Distribusi {TARGET_COLUMN}")
    plt.xlabel(TARGET_COLUMN)
    plt.ylabel("Jumlah")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()

## 9. Pemeriksaan hubungan awal antarvariabel

Korelasi Pearson ditampilkan hanya untuk variabel numerik. Korelasi tidak membuktikan hubungan sebab-akibat dan dapat tidak sesuai untuk variabel kategori yang dikodekan sebagai angka.

In [ ]:
correlation_cols = [c for c in numeric_cols if df[c].nunique(dropna=True) > 1]

if len(correlation_cols) >= 2:
    corr = df[correlation_cols].corr(method="pearson")
    fig, ax = plt.subplots(figsize=(max(8, len(correlation_cols) * 0.8), max(6, len(correlation_cols) * 0.65)))
    image = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(correlation_cols)))
    ax.set_yticks(range(len(correlation_cols)))
    ax.set_xticklabels(correlation_cols, rotation=45, ha="right")
    ax.set_yticklabels(correlation_cols)
    for i in range(len(correlation_cols)):
        for j in range(len(correlation_cols)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
    fig.colorbar(image, ax=ax, label="Korelasi Pearson")
    ax.set_title("Korelasi Pearson antarvariabel numerik")
    plt.tight_layout()
    plt.show()
else:
    print("Korelasi tidak dibuat karena kurang dari dua variabel numerik yang bervariasi.")

## 10. Ringkasan dan ekspor hasil eksplorasi

Ringkasan kualitas data diekspor ke folder `reports/eda/`. File keluaran merupakan artefak hasil eksplorasi, bukan pengganti dataset sumber. Jika notebook dijalankan ulang, file ringkasan akan diperbarui.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary = {
    "dataset": DATASET_PATH.name,
    "jumlah_baris": int(df.shape[0]),
    "jumlah_kolom": int(df.shape[1]),
    "jumlah_missing_total": int(df.isna().sum().sum()),
    "jumlah_baris_duplikat": duplicate_rows,
    "kolom_numerik": numeric_cols,
    "kolom_kategorikal": categorical_cols,
    "kolom_konstan": constant_columns,
    "kolom_potensi_identifier": id_like_columns,
    "kandidat_kolom_target": TARGET_COLUMN,
}

summary_path = OUTPUT_DIR / "ringkasan_eda_awal.json"
missing_path = OUTPUT_DIR / "missing_values.csv"
structure_path = OUTPUT_DIR / "struktur_dataset.csv"

with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, ensure_ascii=False, indent=4)

missing.to_csv(missing_path, index=True)
structure.to_csv(structure_path, index=False)

display(Markdown("### Ringkasan eksplorasi"))
display(pd.DataFrame({
    "Pemeriksaan": [
        "Nama dataset", "Jumlah baris", "Jumlah kolom",
        "Total missing values", "Baris duplikat",
        "Kolom numerik", "Kolom kategorikal", "Kandidat target"
    ],
    "Hasil": [
        DATASET_PATH.name, f"{df.shape[0]:,}", f"{df.shape[1]:,}",
        f"{df.isna().sum().sum():,}", f"{duplicate_rows:,}",
        len(numeric_cols), len(categorical_cols), TARGET_COLUMN or "Belum terdeteksi"
    ]
}))

print("File ringkasan berhasil disimpan:")
print(f"- {summary_path}")
print(f"- {missing_path}")
print(f"- {structure_path}")

## 11. Catatan untuk tahap berikutnya

Berdasarkan hasil notebook ini, tim dapat menentukan pekerjaan lanjutan secara terukur:

1. **Data cleaning:** menangani missing values, duplikasi, inkonsistensi kategori, dan nilai yang tidak masuk akal setelah aturan validasi disepakati.
2. **Data dictionary:** mendokumentasikan definisi, tipe, satuan, rentang yang diharapkan, dan sumber setiap variabel.
3. **Feature engineering:** membentuk fitur turunan hanya jika definisi dan dasar perhitungannya jelas.
4. **Analisis DA:** menguji distribusi dan hubungan variabel dengan metode yang sesuai.
5. **Data Science:** menetapkan target, fitur, strategi pembagian data, dan baseline model setelah kualitas data dinilai.

**Catatan metodologis:** dataset ini berkaitan dengan kesehatan ibu dan anak. Jangan menyimpulkan status gizi, anemia, atau diagnosis diare hanya dari ambang yang belum disepakati dengan pedoman kesehatan yang relevan. Pastikan juga asal-usul dan status data (primer, sekunder, atau sintetis) didokumentasikan sebelum menarik kesimpulan substantif.